# Model Exploration: HistGradientBoostingRegressor vs LightGBM

Both models are tuned with Optuna (same CV strategy, same search budget) then evaluated on val/test with R2, MAE, and RMSE so the comparison is apples-to-apples.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

from dotenv import load_dotenv
import os
load_dotenv()
engine = create_engine(os.environ["POSTGRES"])

train = pd.read_sql("SELECT * FROM train_data", engine)
val = pd.read_sql("SELECT * FROM val_data", engine)
test = pd.read_sql("SELECT * FROM test_data", engine)

X_train = train.drop(columns=['price_log', 'id'], errors='ignore')
y_train = train['price_log']

X_val = val.drop(columns=['price_log', 'id'], errors='ignore')
y_val = val['price_log']

X_test = test.drop(columns=['price_log', 'id'], errors='ignore')
y_test = test['price_log']

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

## 1. HistGradientBoostingRegressor — Optuna Tuning

In [ ]:
import optuna
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error

N_TRIALS = 100
CV = KFold(n_splits=5, shuffle=True, random_state=42)

def hgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
        "max_bins": trial.suggest_int("max_bins", 64, 255),
        "random_state": 42,
    }

    scores = []
    for train_idx, val_idx in CV.split(X_train):
        X_tr, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingRegressor(**params)
        model.fit(X_tr, y_tr)
        preds_log = model.predict(X_v)
        scores.append(mean_absolute_percentage_error(np.expm1(y_v), np.expm1(preds_log)))

    return float(np.mean(scores))

hgb_study = optuna.create_study(direction="minimize", study_name="hgb_tuning")
hgb_study.optimize(hgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best HGB CV R2:", hgb_study.best_value)
print("Best HGB params:", hgb_study.best_params)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

hgb_best_model = HistGradientBoostingRegressor(**hgb_study.best_params, random_state=42)
hgb_best_model.fit(X_train, y_train)

hgb_val_preds = hgb_best_model.predict(X_val)
hgb_test_preds = hgb_best_model.predict(X_test)

hgb_metrics = {
    "model": "HistGradientBoostingRegressor",
    "cv_r2": hgb_study.best_value,
    "val_r2": r2_score(y_val, hgb_val_preds),
    "val_mae": mean_absolute_error(y_val, hgb_val_preds),
    "val_rmse": np.sqrt(mean_squared_error(y_val, hgb_val_preds)),
    "test_r2": r2_score(y_test, hgb_test_preds),
    "test_mae": mean_absolute_error(y_test, hgb_test_preds),
    "test_rmse": np.sqrt(mean_squared_error(y_test, hgb_test_preds)),
}

hgb_metrics

## 2. LightGBM — Optuna Tuning

In [ ]:
from lightgbm import LGBMRegressor, early_stopping
from sklearn.metrics import mean_absolute_percentage_error

def lgbm_objective(trial):
    params = {
        "objective": "mae",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "verbose": -1,
    }

    scores = []
    for train_idx, val_idx in CV.split(X_train):
        X_tr, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_v, y_v)],
            eval_metric='mape',
            callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
        )
        preds_log = model.predict(X_v)
        scores.append(mean_absolute_percentage_error(np.expm1(y_v), np.expm1(preds_log)))

    return float(np.mean(scores))

lgbm_study = optuna.create_study(direction="minimize", study_name="lgbm_tuning")
lgbm_study.optimize(
    lgbm_objective,
    n_trials=N_TRIALS,
    timeout=5000,  # hard cap: 30 min for this tuning run, whichever limit hits first
    show_progress_bar=True,
)

print("Best LightGBM CV R2:", lgbm_study.best_value)
print("Best LightGBM params:", lgbm_study.best_params)

[I 2026-07-06 13:59:24,370] A new study created in memory with name: lgbm_tuning
Best trial: 0. Best value: 0.235814:   1%|          | 1/100 [00:09<15:06,  9.16s/it, 9.16/3000 seconds]

[I 2026-07-06 13:59:33,532] Trial 0 finished with value: 0.23581430060159905 and parameters: {'learning_rate': 0.013591003649971352, 'n_estimators': 275, 'num_leaves': 85, 'max_depth': 10, 'min_child_samples': 75, 'subsample': 0.9879789797068984, 'colsample_bytree': 0.8111210550854913, 'reg_alpha': 0.16417054906270942, 'reg_lambda': 5.445039529386418e-07}. Best is trial 0 with value: 0.23581430060159905.


Best trial: 0. Best value: 0.235814:   2%|▏         | 2/100 [00:20<17:07, 10.49s/it, 20.58/3000 seconds]

[I 2026-07-06 13:59:44,949] Trial 1 finished with value: 0.2414415845795721 and parameters: {'learning_rate': 0.007899867800321762, 'n_estimators': 420, 'num_leaves': 80, 'max_depth': 13, 'min_child_samples': 94, 'subsample': 0.9818176546750434, 'colsample_bytree': 0.9555623119206109, 'reg_alpha': 0.1057362556697564, 'reg_lambda': 2.3057092474174392e-07}. Best is trial 0 with value: 0.23581430060159905.


Best trial: 2. Best value: 0.226191:   3%|▎         | 3/100 [00:31<16:58, 10.50s/it, 31.10/3000 seconds]

...

[I 2026-07-06 14:48:10,248] Trial 61 finished with value: 0.21057195193115774 and parameters: {'learning_rate': 0.046134268780905596, 'n_estimators': 1362, 'num_leaves': 146, 'max_depth': 13, 'min_child_samples': 8, 'subsample': 0.897483088021199, 'colsample_bytree': 0.6124786276902965, 'reg_alpha': 0.0005533253377380491, 'reg_lambda': 3.2737551363831424e-06}. Best is trial 31 with value: 0.20962926472467455.


Best trial: 31. Best value: 0.209629:  63%|██████▎   | 63/100 [49:39<37:04, 60.12s/it, 2979.46/3000 seconds]

[I 2026-07-06 14:49:03,835] Trial 62 finished with value: 0.2136639884436557 and parameters: {'learning_rate': 0.053390417289709294, 'n_estimators': 1364, 'num_leaves': 147, 'max_depth': 13, 'min_child_samples': 15, 'subsample': 0.8934948205414036, 'colsample_bytree': 0.6445365804659409, 'reg_alpha': 0.00012091398603808133, 'reg_lambda': 2.3830904593498376e-06}. Best is trial 31 with value: 0.20962926472467455.


Best trial: 31. Best value: 0.209629:  64%|██████▍   | 64/100 [1:01:42<34:42, 57.85s/it, 3702.17/3000 seconds]   

[I 2026-07-06 15:01:06,534] Trial 63 finished with value: 0.2142275173781159 and parameters: {'learning_rate': 0.07276373462741657, 'n_estimators': 1290, 'num_leaves': 177, 'max_depth': 14, 'min_child_samples': 12, 'subsample': 0.8695580771396045, 'colsample_bytree': 0.6110630649776188, 'reg_alpha': 0.0013138105520994808, 'reg_lambda': 1.7617123704726022e-06}. Best is trial 31 with value: 0.20962926472467455.
Best LightGBM CV R2: 0.20962926472467455
Best LightGBM params: {'learning_rate': 0.018696343827231725, 'n_estimators': 1191, 'num_leaves': 110, 'max_depth': 12, 'min_child_samples': 5, 'subsample': 0.9457908640268251, 'colsample_bytree': 0.7524340315408312, 'reg_alpha': 0.00010954507540653499, 'reg_lambda': 1.0286332154599968e-05}


In [5]:
lgbm_best_model = LGBMRegressor(**lgbm_study.best_params, random_state=42, verbose=-1)
lgbm_best_model.fit(X_train, y_train)

lgbm_val_preds = lgbm_best_model.predict(X_val)
lgbm_test_preds = lgbm_best_model.predict(X_test)

lgbm_metrics = {
    "model": "LightGBM",
    "cv_r2": lgbm_study.best_value,
    "val_r2": r2_score(y_val, lgbm_val_preds),
    "val_mae": mean_absolute_error(y_val, lgbm_val_preds),
    "val_rmse": np.sqrt(mean_squared_error(y_val, lgbm_val_preds)),
    "test_r2": r2_score(y_test, lgbm_test_preds),
    "test_mae": mean_absolute_error(y_test, lgbm_test_preds),
    "test_rmse": np.sqrt(mean_squared_error(y_test, lgbm_test_preds)),
}

lgbm_metrics

NameError: name 'r2_score' is not defined

## 3. Model Comparison

In [ ]:
comparison_df = pd.DataFrame([hgb_metrics, lgbm_metrics]).set_index("model")
comparison_df = comparison_df.round(4)
comparison_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_to_plot = [("test_r2", "Test R2 (higher is better)"),
                    ("test_mae", "Test MAE (lower is better)"),
                    ("test_rmse", "Test RMSE (lower is better)")]

for ax, (col, title) in zip(axes, metrics_to_plot):
    comparison_df[col].plot(kind="bar", ax=ax, color=["#2E86AB", "#F18F01"])
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

**Winner:** whichever model has the higher `test_r2` and lower `test_mae` / `test_rmse` — check `comparison_df` above. If they're close, prefer LightGBM for inference speed, or HGB for fewer dependencies (it's built into sklearn).